# SME pump & dump study: full day-point history, clusters, conditional probabilities

**What this notebook does**
1. Reads your `SME_IPO_Database_v8.xlsx` (Main Database tab) for tickers, listing dates, issue prices, BRLMs and anchors.
2. Downloads the daily **bhavcopy** files from NSE (alphabetic tickers) and BSE (numeric scrip codes) and extracts each IPO's daily close. Yahoo is not used.
3. **Applies your day rule** (Day N = exact calendar day; if shut, next trading day within 10 days) and checks whether Day 1 is the listing date or the day after by matching the downloaded closes to your Day 7/30/60/90 columns, then builds every day-point you asked for: 7, 15, 21, 30, 35, 45, 60, 67, 75, 81, 90, 100, 120, 150, 180, 270, 365, 400.
4. Exports a clean `% return` table with the proper listing date, clusters the IPOs, prints the conditional-probability tree with confidence intervals and an anchor vs non-anchor test, runs an event study around the Day-30 lock-in expiry against placebo windows, and finds lookalikes.

**How to run:** easiest is Google Colab (upload this notebook and the database file, then Runtime > Run all). Locally: `pip install pandas openpyxl matplotlib scipy requests`. The first run downloads about 2 files per trading day since Jan 2025 and takes roughly 15 to 30 minutes; files are cached in `bhav_cache2/` so re-runs are instant. Cell 4 is a probe that prints the HTTP status per exchange, so you can see immediately whether NSE and BSE downloads work from Colab.

**Check these two outputs before trusting anything else:** the *convention test table* (cell 5) and the *fetch coverage list* (cell 4). Where a stock cannot be fetched the notebook falls back to the values already in your database.

In [ ]:
# !pip install -q pandas openpyxl matplotlib scipy requests
import os, time, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import fisher_exact
warnings.filterwarnings("ignore")

import glob
DB_PATH = (glob.glob("SME_IPO_Database*.xlsx") + glob.glob("/content/SME_IPO_Database*.xlsx") + [None])[0]
if DB_PATH is None:                                   # not found: ask for the file (works in Google Colab)
    from google.colab import files
    up = files.upload(); DB_PATH = list(up.keys())[0]
print("Using database file:", DB_PATH)
DAY_POINTS = [1,7,15,21,30,35,45,60,67,75,81,90,100,120,150,180,270,365,400]

# Pattern parameters (same defaults as the Excel workbook)
P = dict(band=0.10,      # Day-1 return must be within +/- band of issue price
         wcut=0.0,       # early-weak: Day-7 (and Day-15) return <= wcut
         use_d15=True,   # also require Day-15 <= wcut (needs the price history)
         lift=0.10,      # W1 pump: best(D30,D35)/D7 - 1 >= lift
         sus=0.10, col=0.35,           # fate after W1 vs Day-100: sustained / fade / collapse
         w2setup=0.0, w2lift=0.10,     # W2 (Day 90/100): underwater at D60, lift >= w2lift
         w3setup=0.0, w3lift=0.25,     # W3 (Day 365/400): underwater at D100, lift >= w3lift
         match_band=0.10, min_frac=0.8)# lookalike: within +/- match_band at >= min_frac of day-points
TODAY = pd.Timestamp.today().normalize()

In [ ]:
raw = pd.read_excel(DB_PATH, sheet_name="Main Database", header=0).iloc[1:].copy()
raw.columns = [str(c).strip() for c in raw.columns]
raw = raw.rename(columns={raw.columns[0]: "Company"}).dropna(subset=["Company"]).reset_index(drop=True)
db = pd.DataFrame({
    "Company": raw["Company"],
    "Ticker":  raw["Stock Tickers"].astype(str).str.strip().str.replace(r"\.0$", "", regex=True),
    "Listing": pd.to_datetime(raw["Listing Date"], errors="coerce"),
    "Issue":   pd.to_numeric(raw["Issue Price"], errors="coerce"),
    "BRLM":    raw["BRLM"], "Anchors": raw["Names of Anchors"]})
for n, c in {1:"Day-1 Close",7:"Day 7 Close",30:"Day 30 Close",35:"Day 35 Close",60:"Day 60 Close",
             90:"Day 90 Close",100:"Day 100 Close",365:"Day 365 Close",400:"Day 400 Close"}.items():
    db[f"db{n}"] = pd.to_numeric(raw[c], errors="coerce")
db["anchor"] = db["Anchors"].notna()
print(len(db), "IPOs;", db.Listing.min().date(), "to", db.Listing.max().date())

In [ ]:
# ---- Price source: daily bhavcopy files published by NSE and BSE (Yahoo is NOT used any more)
import requests, io, zipfile
S = requests.Session()
S.headers.update({"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36",
                  "Accept": "*/*", "Accept-Language": "en-US,en;q=0.9"})
def url_list(ex, d):
    if ex == "BSE":
        return [f"https://www.bseindia.com/download/BhavCopy/Equity/BhavCopy_BSE_CM_0_0_0_{d:%Y%m%d}_F_0000.CSV",
                f"https://www.bseindia.com/download/BhavCopy/Equity/BSE_EQ_BHAVCOPY_{d:%d%m%Y}.ZIP",
                f"https://www.bseindia.com/download/BhavCopy/Equity/EQ{d:%d%m%y}_CSV.ZIP"]
    return [f"https://nsearchives.nseindia.com/content/cm/BhavCopy_NSE_CM_0_0_0_{d:%Y%m%d}_F_0000.csv.zip",
            f"https://nsearchives.nseindia.com/products/content/sec_bhavdata_full_{d:%d%m%Y}.csv"]

def parse(content, ex):
    try:
        if content[:2] == b"PK":
            z = zipfile.ZipFile(io.BytesIO(content)); content = z.read(z.namelist()[0])
        df = pd.read_csv(io.BytesIO(content)); df.columns = [str(c).strip() for c in df.columns]
        idc = next((c for c in (["FinInstrmId", "SC_CODE"] if ex == "BSE" else ["TckrSymb", "SYMBOL"]) if c in df.columns), None)
        cc = next((c for c in ["ClsPric", "CLOSE", "CLOSE_PRICE"] if c in df.columns), None)
        if idc is None or cc is None: return None
        out = pd.DataFrame({"id": df[idc].astype(str).str.strip().str.replace(r"\.0$", "", regex=True),
                            "close": pd.to_numeric(df[cc], errors="coerce")})
        return out.dropna()
    except Exception:
        return None

os.makedirs("bhav_cache2", exist_ok=True)
NEED = {"BSE": set(t for t in db.Ticker if t.isdigit()), "NSE": set(t for t in db.Ticker if not t.isdigit())}
LOG = []
def day_data(ex, d):
    fn = f"bhav_cache2/{ex}_{d:%Y%m%d}.csv"
    if os.path.exists(fn): return pd.read_csv(fn, dtype={"id": str})
    res, status = None, []
    for u in url_list(ex, d):
        try:
            r = S.get(u, timeout=30, headers={"Referer": "https://www.bseindia.com/" if ex == "BSE" else "https://www.nseindia.com/"})
            status.append(r.status_code)
            if r.status_code == 200:
                x = parse(r.content, ex)
                if x is not None: x = x[x.id.isin(NEED[ex])]
                if x is not None and len(x): res = x; break      # use the first file that actually contains our tickers
        except Exception as e:
            status.append(type(e).__name__)
    LOG.append((ex, d, status))
    if res is None:
        if status and all(s == 404 for s in status) and d < TODAY - pd.Timedelta(days=3):
            pd.DataFrame(columns=["id", "close"]).to_csv(fn, index=False)      # holiday: remember (never for the last few days)
        return pd.DataFrame(columns=["id", "close"])
    res.to_csv(fn, index=False); return res

# PROBE: try the last 6 weekdays on both exchanges and show what happens
for ex in ("NSE", "BSE"):
    for d in pd.bdate_range(end=TODAY, periods=6):
        got = day_data(ex, d); print(ex, d.date(), "rows kept:", len(got), "| http:", LOG[-1][2] if LOG and LOG[-1][1] == d and LOG[-1][0] == ex else "cached")

In [ ]:
dates = pd.bdate_range(db.Listing.min() - pd.Timedelta(days=5), TODAY)
frames = []
for i, d in enumerate(dates):
    for ex in ("NSE", "BSE"):
        x = day_data(ex, d)
        if len(x): frames.append(x.assign(date=d, ex=ex))
    if i % 50 == 0: print(f"{i}/{len(dates)} dates done", end="\r"); time.sleep(0.2)
allp = pd.concat(frames) if frames else pd.DataFrame(columns=["id", "close", "date", "ex"])
bykey = {k: g.set_index("date")["close"].sort_index() for k, g in allp.groupby(["ex", "id"])}
prices, used = {}, {}
for _, r in db.iterrows():
    ex = "BSE" if r.Ticker.isdigit() else "NSE"
    s = bykey.get((ex, r.Ticker))
    if s is not None and pd.notna(r.Listing):
        s = s[s.index >= r.Listing]
        if len(s): prices[r.Company] = s; used[r.Company] = f"{ex}:{r.Ticker}"
missing = [c for c in db.Company if c not in prices]
print(f"fetched {len(prices)} of {len(db)}; {len(missing)} not found (these fall back to database values)")
print(missing[:60])

In [ ]:
# Your rule: Day N = exact calendar day (listing date + N days); if the market was shut that day,
# take the NEXT available close within MAX_FWD calendar days. The test below only checks whether
# "Day 1" is the listing date itself (offset N-1) or listing date + 1 (offset N).
MAX_FWD = 10
CONV = ["calendar_N-1", "calendar_N"]
def day_close(x, listing, n, conv, max_fwd=MAX_FWD):
    off = n - 1 if conv.endswith("N-1") else n
    t0 = listing + pd.Timedelta(days=off)
    y = x[(x.index >= t0) & (x.index <= t0 + pd.Timedelta(days=max_fwd))]
    return y.iloc[0] if len(y) else np.nan

rows = []
for conv in CONV:
    hits = tot = 0
    for _, r in db.iterrows():
        x = prices.get(r.Company)
        if x is None: continue
        for n in (7, 30, 60, 90):
            v = r[f"db{n}"]
            f = day_close(x, r.Listing, n, conv)
            if pd.isna(v) or pd.isna(f): continue
            tot += 1; hits += abs(f / v - 1) < 0.01
    rows.append((conv, int(hits), tot, round(hits / max(tot, 1), 3)))
conv_df = pd.DataFrame(rows, columns=["convention", "matches(<1% diff)", "compared", "match_rate"])
display(conv_df) if "display" in globals() else print(conv_df)
BEST = conv_df.sort_values("match_rate").iloc[-1].convention
d1 = [abs(prices[r.Company].iloc[0] / r.db1 - 1) < 0.01 for _, r in db.iterrows() if r.Company in prices and pd.notna(r.db1)]
print("Best convention:", BEST, "| Day-1 close matches database for", f"{np.mean(d1):.0%}" if d1 else "n/a", "of stocks")

In [ ]:
recs = []
for _, r in db.iterrows():
    x = prices.get(r.Company)
    d = {"Company": r.Company, "Ticker": r.Ticker, "Listing Date": r.Listing, "Issue": r.Issue,
         "BRLM": r.BRLM, "Anchor-backed": r.anchor, "Source": used.get(r.Company)}
    for n in DAY_POINTS:
        v = np.nan if x is None else day_close(x, r.Listing, n, BEST)
        if pd.isna(v) and f"db{n}" in r.index: v = r[f"db{n}"]        # fallback to your database
        d[n] = v
    d["Today"] = x.iloc[-1] if x is not None else np.nan
    recs.append(d)
px = pd.DataFrame(recs)
ret = px[["Company"]].copy()
for n in DAY_POINTS + ["Today"]: ret[n] = px[n] / px["Issue"] - 1

out = px[["Company", "Ticker", "Listing Date", "Issue", "BRLM", "Anchor-backed"]].copy()
for n in DAY_POINTS + ["Today"]: out[f"Day {n} %" if n != "Today" else "Today %"] = ret[n]
with pd.ExcelWriter("Day_Points_Returns.xlsx", engine="openpyxl", datetime_format="dd-mmm-yyyy") as w:
    out.to_excel(w, sheet_name="Returns_vs_Issue", index=False)
    px.to_excel(w, sheet_name="Prices", index=False)
    ws = w.sheets["Returns_vs_Issue"]
    for col in ws.iter_cols(min_col=7, min_row=2):
        for c in col: c.number_format = "0.0%"
print("wrote Day_Points_Returns.xlsx", out.shape)

# ---- completeness report: which stocks are fully covered, and what fell back to your database
due = lambda r, n: r["Listing Date"] + pd.Timedelta(days=n + MAX_FWD) <= TODAY
cov = []
for _, r in px.iterrows():
    x = prices.get(r.Company); need = [n for n in DAY_POINTS if due(r, n)]
    have = [n for n in need if pd.notna(r[n])]
    yh = [n for n in need if x is not None and pd.notna(day_close(x, r["Listing Date"], n, BEST))]
    d1ok = (x is not None) and abs(x.iloc[0] / db.set_index("Company").db1.get(r.Company, np.nan) - 1) < 0.01
    cov.append({"Company": r.Company, "Source": r["Source"], "Fetched": x is not None, "Price rows": 0 if x is None else len(x),
                "Day-1 matches DB": bool(d1ok), "Day-points due": len(need), "Filled (any source)": len(have), "Filled from exchange files": len(yh),
                "Missing day-points": ", ".join(str(n) for n in need if n not in have),
                "Status": "complete" if len(have) == len(need) else ("partial" if have else "no data")})
cov = pd.DataFrame(cov)
print(cov.Status.value_counts().to_string())
with pd.ExcelWriter("Day_Points_Returns.xlsx", engine="openpyxl", mode="a", if_sheet_exists="replace") as w:
    cov.to_excel(w, sheet_name="Coverage", index=False)
print("Coverage sheet added. Open it and filter Status != complete: those IPOs need manual price entry.")

In [ ]:
def classify(ret, px, P):
    d = ret.set_index("Company").copy(); d.columns = [f"r{c}" for c in d.columns]
    p = px.set_index("Company")
    weak = (d["r1"].abs() <= P["band"]) & (d["r7"] <= P["wcut"])
    if P["use_d15"]: weak &= (d["r15"] <= P["wcut"])
    d["early"] = weak.fillna(False)
    d["anchor"] = p["Anchor-backed"].astype(bool); d["BRLM"] = p["BRLM"]
    d["w1peak"] = p[[30, 35]].max(axis=1); d["w1lift"] = d["w1peak"] / p[7] - 1
    d["w1"] = d["early"] & (d["w1lift"] >= P["lift"])
    ref = p[100].fillna(p[90]); d["w1post"] = ref / d["w1peak"] - 1
    d["w1fate"] = np.where(~d["w1"], "", np.where(d["w1post"].isna(), "No data",
                  np.where(d["w1post"] >= -P["sus"], "Sustained", np.where(d["w1post"] > -P["col"], "Fade", "Collapse"))))
    d["w2peak"] = p[[90, 100]].max(axis=1); d["w2lift"] = d["w2peak"] / p[60] - 1
    d["w2"] = (d["r60"] < P["w2setup"]) & (d["w2lift"] >= P["w2lift"])
    d["w3peak"] = p[[365, 400]].max(axis=1); d["w3lift"] = d["w3peak"] / p[100] - 1
    d["w3"] = (d["r100"] < P["w3setup"]) & (d["w3lift"] >= P["w3lift"])
    d["w2post"] = p[365] / d["w2peak"] - 1; d["w3post"] = p["Today"] / d["w3peak"] - 1
    d["cluster"] = np.where(d["w1"], "W1 " + d["w1fate"], np.where(d["early"], "Early-weak, no pump", "Other"))
    return d
D = classify(ret, px, P)

def wilson(k, n, z=1.96):
    if n == 0: return (np.nan, np.nan, np.nan)
    p = k / n; den = 1 + z*z/n
    c = (p + z*z/(2*n)) / den; h = z*np.sqrt(p*(1-p)/n + z*z/(4*n*n)) / den
    return p, c - h, c + h

def row(label, k, n): p, lo, hi = wilson(int(k), int(n)); return dict(step=label, k=int(k), n=int(n), prob=p, lo=lo, hi=hi)
e = D[D.early]; ok = e.w1lift.notna()
tree = [row("Early-weak share of IPOs with D1/D7 data", D.early.sum(), D.r7.notna().sum()),
        row("W1 pump | early-weak", e.w1.sum(), ok.sum()),
        row("  anchor-backed", e[e.anchor & ok].w1.sum(), (e.anchor & ok).sum()),
        row("  non-anchor", e[~e.anchor & ok].w1.sum(), (~e.anchor & ok).sum())]
pump = D[D.w1 & (D.w1fate != "No data")]
for f in ["Sustained", "Fade", "Collapse"]: tree.append(row(f"  fate after W1: {f}", (pump.w1fate == f).sum(), len(pump)))
el2 = D.r60 < P["w2setup"]; el2 &= D.w2lift.notna()
tree += [row("W2 pump | underwater at D60 (all)", D.w2[el2].sum(), el2.sum()),
         row("W2 pump | underwater at D60, early-weak", D.w2[el2 & D.early].sum(), (el2 & D.early).sum()),
         row("W2 pump | underwater at D60, not early-weak", D.w2[el2 & ~D.early].sum(), (el2 & ~D.early).sum())]
el3 = D.r100 < P["w3setup"]; el3 &= D.w3lift.notna()
tree += [row("W3 pump | underwater at D100 (all)", D.w3[el3].sum(), el3.sum()),
         row("W3 pump | underwater at D100, early-weak", D.w3[el3 & D.early].sum(), (el3 & D.early).sum())]
tree = pd.DataFrame(tree)
for c in ["prob", "lo", "hi"]: tree[c] = (tree[c] * 100).round(1)
print(tree.to_string(index=False))
a, n_ = e[e.anchor & ok], e[~e.anchor & ok]
odds, pf = fisher_exact([[a.w1.sum(), len(a) - a.w1.sum()], [n_.w1.sum(), len(n_) - n_.w1.sum()]])
print(f"Fisher exact anchor vs non-anchor pump rate: odds ratio {odds:.2f}, p = {pf:.3f}")

In [ ]:
# Average % return trajectory per cluster at every day-point (mean and median)
cols = [n for n in DAY_POINTS] + ["Today"]
traj_mean = D.groupby("cluster")[[f"r{c}" for c in cols]].mean().T
traj_med  = D.groupby("cluster")[[f"r{c}" for c in cols]].median().T
traj_n    = D.groupby("cluster").size()
print(traj_n.to_string()); display(traj_med.round(3)) if "display" in globals() else print(traj_med.round(3))

W1 = D[D.w1]
xs = [n for n in DAY_POINTS if n <= 100]; tmpl = W1[[f"r{n}" for n in xs]].mean()
fig, ax = plt.subplots(figsize=(11, 5.5))
for _, r in W1.iterrows(): ax.plot(xs, [r[f"r{n}"] * 100 for n in xs], color="grey", alpha=.25, lw=1)
ax.plot(xs, tmpl.values * 100, "o-", color="#1F3864", lw=2.5, label=f"W1 pump average (n={len(W1)})")
ax.fill_between(xs, (tmpl.values - P["match_band"]) * 100, (tmpl.values + P["match_band"]) * 100, color="#1F3864", alpha=.12, label=f"+/-{P['match_band']:.0%} band")
for d_, t_ in [(30, "Anchor 50% unlock"), (90, "Anchor 100% unlock")]: ax.axvline(d_, ls="--", color="#C00000", alpha=.6); ax.text(d_ + 1, ax.get_ylim()[1] * .9, t_, color="#C00000", fontsize=8)
ax.axhline(0, color="k", lw=.8); ax.set_xlabel("Day since listing"); ax.set_ylabel("% return vs issue price")
ax.set_title("Window-1 pump cohort: individual paths vs cohort average"); ax.legend(); plt.tight_layout(); plt.savefig("w1_trajectory.png", dpi=150); plt.show()

In [ ]:
# Event study around the Day-30 lock-in expiry: calendar-day paths indexed to calendar day 20
def at(x, listing, off):
    cut = listing + pd.Timedelta(days=off); y = x[x.index <= cut]
    return y.iloc[-1] if len(y) and cut <= x.index[-1] else np.nan
def path(x, listing, base, offs): b = at(x, listing, base); return np.array([at(x, listing, o) / b - 1 for o in offs])
offs = list(range(20, 46))
def group_paths(mask):
    L = [path(prices[c], db.set_index("Company").Listing[c], 20, offs) for c in D[mask].index if c in prices]
    return np.array([p for p in L if not np.isnan(p).any()])
A, N_ = group_paths(D.early & D.anchor), group_paths(D.early & ~D.anchor)
rng = np.random.default_rng(7)
def boot(M, B=2000):
    if len(M) < 3: return None
    m = np.array([M[rng.integers(0, len(M), len(M))].mean(0) for _ in range(B)]); return np.percentile(m, [5, 95], axis=0)
fig, ax = plt.subplots(figsize=(11, 5))
for M, lab, col in [(A, "Early-weak, anchor-backed", "#1F3864"), (N_, "Early-weak, no anchors", "#C00000")]:
    if len(M) == 0: continue
    ax.plot(offs, M.mean(0) * 100, color=col, lw=2, label=f"{lab} (n={len(M)})"); b = boot(M)
    if b is not None: ax.fill_between(offs, b[0] * 100, b[1] * 100, color=col, alpha=.12)
ax.axvspan(27, 35, color="orange", alpha=.15, label="expected unlock window"); ax.axhline(0, color="k", lw=.8)
ax.set_xlabel("Calendar day since listing"); ax.set_ylabel("Cumulative % vs calendar-day 20"); ax.legend(); ax.set_title("Event study: price path into the Day-30 anchor unlock"); plt.tight_layout(); plt.savefig("event_study.png", dpi=150); plt.show()

# Placebo test: is the 27->35 window return unusual compared with other 8-day windows for the same stocks?
def win(M_ids, a, b): return np.nanmean([at(prices[c], db.set_index("Company").Listing[c], b) / at(prices[c], db.set_index("Company").Listing[c], a) - 1 for c in M_ids if c in prices])
ids = list(D[D.early & D.anchor].index)
actual = win(ids, 27, 35); null = np.array([win(ids, s, s + 8) for s in range(40, 120, 4)])
print(f"anchor-backed early-weak: mean return day 27->35 = {actual:.2%}; placebo windows mean {np.nanmean(null):.2%}, "
      f"actual percentile within placebo = {(null < actual).mean():.0%}")

In [ ]:
# Lookalikes: IPOs outside the W1 cohort whose path stays inside the template band
tp = tmpl; pts = [f"r{n}" for n in xs]
def matchfrac(r):
    v = r[pts].astype(float); ok = v.notna()
    return np.nan if ok.sum() < 4 else (np.abs(v[ok] - tp.values[ok.values]) <= P["match_band"]).mean()
D["match"] = D.apply(matchfrac, axis=1)
look = D[(D.match >= P["min_frac"]) & ~D.w1].sort_values("match", ascending=False)
print(len(look), "lookalike IPOs")
res = look[["BRLM", "anchor", "early", "match"] + pts + ["rToday"]].round(3); print(res.head(30).to_string())
for i, c in enumerate(look.index[:6]):
    plt.plot(xs, [look.loc[c, f"r{n}"] * 100 for n in xs], label=c[:28], alpha=.8)
plt.plot(xs, tp.values * 100, "k", lw=3, label="W1 template"); plt.legend(fontsize=7); plt.title("Closest lookalikes"); plt.savefig("lookalikes.png", dpi=150); plt.show()

with pd.ExcelWriter("Clusters_and_Probabilities.xlsx", engine="openpyxl") as w:
    tree.to_excel(w, sheet_name="CondProb", index=False)
    D.reset_index().to_excel(w, sheet_name="Clusters", index=False)
    traj_mean.to_excel(w, sheet_name="Trajectory_mean"); traj_med.to_excel(w, sheet_name="Trajectory_median")
    res.reset_index().to_excel(w, sheet_name="Lookalikes", index=False)
print("done: Day_Points_Returns.xlsx, Clusters_and_Probabilities.xlsx, PNG charts")